In [0]:
g_env = 'DEV'
g_ucBronze = 'dev_hub_bronze'
v_p_srcSchema = 'lh_ax_idr'
g_ucSilver = 'dev_hub_silver'

In [0]:
import pyspark.sql.functions as f
from pyspark.sql.functions import lit, col
import json
import time

##extract using etteration

In [0]:
# pk_dict = dbutils.widgets.get("input")
# print(pk_dict)


In [0]:
pk_dict = ["tcrstat", "tcrrlcl", "tcrroo1", "tcrrhdr"]

In [0]:
# pk_dict = dbutils.jobs.taskValues.get(taskKey="pk_dict", default=None)
df_meta = spark.table(f'dev_bronze.poc._meta')
pk_cols_map = {}

for table in pk_dict:

    pk_cols = df_meta.filter(col("TABLE_NM") == table.upper()).select("PK_COL_NM").distinct().collect()
    pk_cols_map[table] = pk_cols
print(pk_cols_map)    

In [0]:
def sync_idr_tables(table_list):
    pk_cols_map_str = {
        table: [row['PK_COL_NM'] for row in pk_cols]
        for table, pk_cols in pk_cols_map.items()
    }
    for table, pk_cols in pk_cols_map_str.items():

        sdf = spark.table(f'{g_ucBronze}.{v_p_srcSchema}.{table}')
        sdf.createOrReplaceTempView('sdf')
        pk_cols_str = ', '.join(pk_cols)


        # Deduplicate source
        deduped = spark.sql(f'''
            SELECT *
            FROM (
                SELECT *,
                    ROW_NUMBER() OVER (PARTITION BY {pk_cols_str}
                                        ORDER BY data_received_utc_dttm DESC) AS rn
                FROM sdf) t
            WHERE rn = 1
        ''')
        final = deduped.drop("rn")
        final.createOrReplaceTempView("final")
        # Get all columns from the DataFrame
        columns = final.columns
        columns_str = ', '.join(columns)
        values_str = ', '.join([f's.{col}' for col in columns])
    #   Ingest target table with deduplicated data
        merge_qr = f"""
            MERGE WITH SCHEMA EVOLUTION
                    INTO {g_ucSilver}.{v_p_srcSchema}.{table} t
                    USING final s
                        ON {" AND ".join([f"t.{col} = s.{col}" for col in pk_cols])}
                        WHEN MATCHED THEN UPDATE SET {', '.join([f"t.{col} = s.{col}" for col in columns if col not in pk_cols])}
                        WHEN NOT MATCHED THEN INSERT ({columns_str}) VALUES ({values_str})
        """
        ingest = spark.sql(merge_qr)
        #Summary
        print(f'Merge Completed Successfully')





In [0]:
p_maxWorkers = 4
import json

pk_json = json.dumps(pk_cols_map)
print(pk_cols_map)


In [0]:

from concurrent.futures import ThreadPoolExecutor
with ThreadPoolExecutor(max_workers=p_maxWorkers) as ex:
    start_time = time.time()
    sync_idr_tables(pk_json)
    end_time = time.time()
    print(f"Execution time: {end_time - start_time:.2f} seconds")